# Medical Word Detection

**Run Cell 1 (load model) ONCE.** Then re-run the detection cells as many times as you like - the model stays loaded in memory, so detection is instant.

Pipeline: spell-check GATE first (only non-words reach the model) -> BioClinicalBERT scores the candidates.

## Cell 1 - Imports + load the model (run ONCE)

In [ ]:
import pandas as pd
import json
import re
import torch

from spellchecker import SpellChecker
from transformers import AutoTokenizer, AutoModelForMaskedLM

MODEL_LABEL = "BioClinicalBERT"
MODEL_NAME = "emilyalsentzer/Bio_ClinicalBERT"

# Use GPU automatically if available (much faster).
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading {MODEL_LABEL} on {device} ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForMaskedLM.from_pretrained(MODEL_NAME).to(device)
model.eval()

spell = SpellChecker()

print("Model loaded. You can now run the detection cells repeatedly without reloading.")

## Cell 2 - Config + detection functions (run once; re-run if you edit them)

In [ ]:
SUSPICION_THRESHOLD = 0.0001   # among non-words, how unlikely to count as wrong
MIN_WORD_LENGTH = 1
HL_OPEN, HL_CLOSE = "<<", ">>"


def is_real_word(clean_word):
    """True if the spell-checker recognises it as a real word."""
    return len(spell.unknown([clean_word.lower()])) == 0


def word_probability(plain_words, index, clean_word):
    """Mask the word (correct number of [MASK] tokens) in the full
    paragraph and return the model's average probability for it."""
    word_token_ids = tokenizer(clean_word, add_special_tokens=False)["input_ids"]
    n_tokens = len(word_token_ids)
    if n_tokens == 0:
        return None

    masked = plain_words.copy()
    masked[index] = " ".join([tokenizer.mask_token] * n_tokens)
    masked_sentence = " ".join(masked)

    inputs = tokenizer(masked_sentence, return_tensors="pt", truncation=True, max_length=512).to(device)
    mask_positions = (inputs["input_ids"][0] == tokenizer.mask_token_id).nonzero(as_tuple=True)[0]
    if len(mask_positions) == 0:
        return None

    with torch.no_grad():
        logits = model(**inputs).logits
    probs = torch.softmax(logits[0], dim=-1)

    usable = min(n_tokens, len(mask_positions))
    token_probs = [probs[mask_positions[k].item(), word_token_ids[k]].item() for k in range(usable)]
    return sum(token_probs) / len(token_probs) if token_probs else None


def detect_incorrect_words(text):
    """Gate first (skip real words), then score only the non-words."""
    detections = []
    word_spans = [(m.group(), m.start(), m.end()) for m in re.finditer(r"\S+", text)]
    plain_words = [w for (w, _s, _e) in word_spans]

    for i, (raw_word, start, end) in enumerate(word_spans):
        clean_word = re.sub(r"[^A-Za-z]", "", raw_word)
        if len(clean_word) < MIN_WORD_LENGTH:
            continue
        if is_real_word(clean_word):        # GATE: skip real words
            continue
        prob = word_probability(plain_words, i, clean_word)
        if prob is None:
            continue
        if prob < SUSPICION_THRESHOLD:
            detections.append({
                "word": raw_word, "start": start, "end": end,
                "detection_confidence": round(1.0 - prob, 6)
            })
    return detections


def highlight_paragraph(text, detections):
    highlighted = text
    for d in sorted(detections, key=lambda x: x["start"], reverse=True):
        s, e = d["start"], d["end"]
        highlighted = highlighted[:s] + HL_OPEN + highlighted[s:e] + HL_CLOSE + highlighted[e:]
    return highlighted


def process_text(text):
    if pd.isna(text):
        return {"highlighted_text": "", "detected_words": [], "combined_confidence": 0.0}
    text = str(text)
    detections = detect_incorrect_words(text)
    combined = sum(d["detection_confidence"] for d in detections) / len(detections) if detections else 0.0
    return {
        "highlighted_text": highlight_paragraph(text, detections),
        "detected_words": detections,
        "combined_confidence": round(combined, 6)
    }

print("Functions ready.")

## Cell 3 - Quick test on one paragraph (instant, re-run freely)

In [ ]:
sample = "the patient is suspected to have tengu. dengue test cbc. abu antibody titer."

result = process_text(sample)

print(result["highlighted_text"])
print()
print(json.dumps(result, indent=2, ensure_ascii=False))

## Cell 4 - Run over the whole Excel file and save

In [ ]:
INPUT_FILE = r"C:\Users\HITESH S ROY\OneDrive\Documents\Medical_Checker\medical_checker.xlsx"
OUTPUT_FILE = r"C:\Users\HITESH S ROY\OneDrive\Documents\Medical_Checker\Output\Detection_BioClinicalBERT_Notebook.xlsx"
INPUT_COLUMN = "error"

df = pd.read_excel(INPUT_FILE)
if INPUT_COLUMN not in df.columns:
    raise Exception(f"Excel file must contain '{INPUT_COLUMN}' column")

outputs = []
for index, row in df.iterrows():
    result = process_text(row[INPUT_COLUMN])
    outputs.append(json.dumps(result, ensure_ascii=False))
    print(f"Row {index + 1}: {len(result['detected_words'])} flagged, conf = {result['combined_confidence']}")

df[f"{MODEL_LABEL}_detection"] = outputs
df.to_excel(OUTPUT_FILE, index=False)
print(f"\nSaved: {OUTPUT_FILE}")